***

# **About Automation**

***

This file is focused on automating the About pages on every excel file that we make for the indicators. The user needs to define the `sample_type`, `indicator_name`, dataframe that the possible years will be pulled from, and the date will be pulled automatically. The code also has the process of converting the `About Indicators.xlsx` file to a YAML type. From the YAML file, `dict_about.yaml`, more edits were made to include additional indicators. From this file, users can indicate the about section of a page by searching the file for the indicator name, and then editing the section dedicated to this indicator. 

The code works by linking to this YAML file, and extracting the information of the specific sample and indicator name the user wants. From there, this subset is converted to a pandas dataframe. This dataframe is parsed through to check for dynamic data (fields that change values regularly), such as year(s) or when the indicator was last updated. Afterwards, the notes are checked and cleaned for visual clarity. The data is returned as a dataframe, this way the user can see what the about page will look like for themselves. 

***

Preparing Workspace

***

In [ ]:
import pandas as pd
import os
import yaml
from datetime import date
import numpy as np
from tqdm import tqdm

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'Census')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_code    = os.path.join(path_git, 'Python Code', 'Census')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')
path_yaml    = os.path.join(path_config0, 'dict_about.yaml')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

About Documentation

***

In [ ]:

try:
    with open(path_yaml, 'r') as yaml_file:
        dict_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
    print(dict_about)
except FileNotFoundError:
    print(f"Error: The file at {path_yaml} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")

dict_about

In [ ]:
estimates = list(dict_about.keys())
estimates.remove('ACS1')

list_samples = []
for estimate in estimates:
    if estimate in ['ACS5', 'BLS']:
        list_samples.extend(dict_about[estimate].keys())
samples = unique(list_samples)


list_indicators = []
for estimate in estimates:
    if estimate in ['ACS5', 'BLS']:
        for sample in samples:
            try:
                list_indicators.extend(dict_about[estimate][sample].keys())
            except: pass
    else:
        list_indicators.extend(dict_about[estimate].keys())
indicators = unique(list_indicators)

print(estimates)
print(samples)
print(indicators)

In [ ]:
list_df_about = []

for estimate in estimates:
    if estimate in ['ACS5', 'BLS']:
        for sample in samples:
            for indicator in indicators:
                try:
                    df_about = pd.DataFrame.from_dict(dict_about[estimate][sample][indicator]).T.reset_index().rename(columns = {'index': 'Indicator', 0: indicator})
                    df_about.loc[df_about['Indicator'] == 'Last Updated', indicator] = date.today().strftime('%Y-%m-%d')
                    
                    notes_row = df_about[df_about['Indicator'] == 'Notes'].copy()
                    notes = notes_row[indicator].values[0]
                    lines = notes.split('\\n')
                    new_rows = [{'Indicator': 'Notes' if i == 0 else '', indicator: line} for i, line in enumerate(lines) if line]
                    new_df = pd.DataFrame(new_rows)
                    df_filtered = df_about[df_about['Indicator'] != 'Notes']
                    df_about = pd.concat([df_filtered, new_df], ignore_index=True)
                    
                    list_df_about.append(df_about)
                    
                except: pass
    else:
        for indicator in indicators:
                try:
                    df_about = pd.DataFrame.from_dict(dict_about[estimate][indicator]).T.reset_index().rename(columns = {'index': 'Indicator', 0: indicator})
                    df_about.loc[df_about['Indicator'] == 'Last Updated', indicator] = date.today().strftime('%Y-%m-%d')
                    
                    notes_row = df_about[df_about['Indicator'] == 'Notes'].copy()
                    notes = notes_row[indicator].values[0]
                    lines = notes.split('\\n')
                    new_rows = [{'Indicator': 'Notes' if i == 0 else '', indicator: line} for i, line in enumerate(lines) if line]
                    new_df = pd.DataFrame(new_rows)
                    df_filtered = df_about[df_about['Indicator'] != 'Notes']
                    df_about = pd.concat([df_filtered, new_df], ignore_index=True)
                    
                    list_df_about.append(df_about)
                    
                except: pass
        
display(list_df_about[0], list_df_about[30])

***

Exporting

***

In [ ]:
print('')
print('Replacing current documentation workbook...')

for df in list_df_about:
    with pd.ExcelWriter(os.path.join(path_main, 'About Indicators.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace') as writer:
        df.to_excel(writer, index = False, sheet_name = df.columns[1], header = False)

print('Success')

Code graveyard

In [ ]:
# # Function / Script to write about pages. 

# # Would probs be better to just make it a script, but func for now

# # path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out) # need to define these

# def write_about(sample_type, indicator_name, year_frame, name_output_xlsx, path_out_xlsx):

#     import numpy as np
#     import re
#     import pandas as pd
#     import os
#     from datetime import date

#     # Define path_self, and path_git here. These paths are still subject to change. 

#     path_yaml = os.path.join(path_self, 'dict_about.yaml')
    
#     try:
#         with open(path_yaml, 'r') as yaml_file:
#             dict_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
#     except FileNotFoundError:
#         print(f"Error: The file at {path_yaml} does not exist.")
#     except Exception as e:
#         print(f"An error occurred: {e}")

#     df_dicto = pd.DataFrame.from_dict(dict_about[sample_type][indicator_name]).T.reset_index().rename(columns = {'index': 'Metadata', 0: 'Description'})

#     def evaluate_expression(expr, context):
#         try:
#             # check each cell for code. Provide context for execution of code
#             return eval(expr, {"np": np, "date": date, **context}) # "geography": geography
#         except Exception as e:
#             print(f"Error: Could not evaluate expression: '{expr}': {e}")
#             return expr  # Returns the string if unable to be passed

#     # Context for the dynamic variables
#     context = {
#         "np": np,
#         "date": date,
#         "df": year_frame  # Do this so that we can pass different frames.
#         #, "geography": geography and we'd have to add geography to the evaluate_expression function. Nothing too crazy tho.
#     }

#     # Apply the function to the about tab. 
#     df_dicto['Description'] = df_dicto['Description'].apply(
#         lambda x: evaluate_expression(x, context) if isinstance(x, str) and ('np.' in x or 'date.' in x or 'df' in x) else x
#     )
    
#     # Split notes into rows, for visual clarity in about
#     def split_notes(df):
#         # Find the row with 'Notes', then use that to take the information
#         notes_row = df[df['Metadata'] == 'Notes'].copy()
#         notes = notes_row['Description'].values[0]
        
#         # Separate based off of NewLines, make the rows with this
#         lines = notes.split('\\n')

#         # Make a blank row past the first one. This way, we don't have to see notes as a cell like 7 times.
#         new_rows = [{'Metadata': 'Notes' if i == 0 else '', 'Description': line} for i, line in enumerate(lines) if line]
        
#         # Create a df from the new separated rows. Drop the old notes row
#         new_df = pd.DataFrame(new_rows)
#         df_filtered = df[df['Metadata'] != 'Notes']
        
#         # Combine original with new rows
#         notes_df = pd.concat([df_filtered, new_df], ignore_index=True)
        
#         return notes_df
    
#     # Finally, we split the notes
#     df_dicto = split_notes(df_dicto)

#     # Display for the user
#     print("Visual representation of the output for:", indicator_name)
#     display(df_dicto)

#     # Might be better to just return the dataframe. That way, the user can see what the output looks like.
#     # Additionally, it could be nice to have it so that if the user needs to edit part of the about themselves, they can do so by the following:
#     # df_dicto[df_dicto['Metadata'] == 'Row Val']['Description'].values[0] == 'CHANGE'
#     # a little iffy on making Geography a dynamic variable, a lot of the workbooks are very descriptive with this field
#     with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
#         df_dicto.to_excel(writer, index = False, sheet_name = 'About')
    

In [ ]:
# # For this one, I tested on Pop_3

# path_pop = "C:\\Users\\jchoy\\Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents\\Data\\Vibrant and Inclusive Places\\People and Community\\Pop and Demographics\\Pop_3 Race\\Pop_3 MPO ACS1.xlsx"

In [ ]:
# # Importing the df that supplies the years
# nayeon = pd.read_excel(path_pop)

# # Both sample_type and indicator_name are variables we reference a lot in other workbooks
# sample_type = 'ACS'

# indicator_name = 'Pop_3'

# # Change this as necessary
# name_output_xlsx = 'Test About Export.xlsx'

# # Might want to get rid of headers: metadata and information on export. 
# write_about(sample_type, indicator_name, nayeon, name_output_xlsx, path_self)